## Task 3, part 6 - Modelling of a 5th, additional model: functional-similarity (STRING) kernel regression

Same kernel-weighted-average structure as model 4, but with a fundamentally different kind of similarity: instead of measuring how similar two perturbations' *experimentally measured* protein log2FC is, this uses STRING's functional/physical association score between the two genes -- pure prior biological knowledge (pathway co-membership, physical interaction, co-expression across many public datasets, text-mining, etc.), completely independent of anything measured in this specific experiment. In principle this could even generate a prediction for a gene that was never profiled at all, as long as it's a known human gene.

We first checked KEGG pathway co-membership (Jaccard similarity of pathway sets) as the similarity source, but only 35/50 of our genes have any KEGG pathway annotation at all (6/10 test genes), so 4 of the 10 test genes would have had to fall back to the plain baseline. STRING covers 46/50 genes with at least one scored association (all 10 test genes included), so we use STRING's combined association score instead.

Note: STRING scores are *not* condition-specific (they reflect prior knowledge about the two genes in general, not about Control/IFNγ/Co-culture specifically) -- unlike the RNA/protein fingerprints, the same similarity matrix is used for all three conditions. This is a simplifying assumption worth stating plainly: two genes' general functional relationship doesn't change between conditions, even though the *size* of their perturbation effect might.

In [1]:
import os
import io
import urllib.request
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()

selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

## Fetch (and cache) STRING functional-association scores

Query STRING's public REST API once for all pairwise association scores among our 50 selected genes (`required_score=0` asks for every scored pair, however weak), then cache the result to a local file. Every later rerun of this notebook loads the cached file instead of hitting the API again -- this is the only notebook in the project that depends on external data/internet access at all, so caching keeps it reproducible offline after the first run.

In [3]:
STRING_CACHE = f"{DATA_DIR}/task3_string_similarity.csv"

if os.path.exists(STRING_CACHE):
    string_edges = pd.read_csv(STRING_CACHE)
else:
    # %0d is STRING's required separator between multiple identifiers in one request
    identifiers = "%0d".join(selected_50)
    url = f"https://string-db.org/api/tsv/network?identifiers={identifiers}&species=9606&required_score=0"
    with urllib.request.urlopen(url, timeout=30) as response:
        text = response.read().decode()
    string_edges = pd.read_csv(io.StringIO(text), sep="\t")[["preferredName_A", "preferredName_B", "score"]]
    string_edges.columns = ["gene_a", "gene_b", "score"]
    string_edges.to_csv(STRING_CACHE, index=False)

string_edges.shape

(238, 3)

## Build a full gene x gene similarity matrix

STRING only returns edges it actually scored; gene pairs with no recorded association at all are simply absent from that list. Build a full, symmetric 50x50 matrix and fill every absent pair with a similarity of 0 -- "no evidence of any functional relationship" is a meaningful value here, not missing data.

In [4]:
string_similarity = pd.DataFrame(0.0, index=selected_50, columns=selected_50)
for _, row in string_edges.iterrows():
    if row["gene_a"] in string_similarity.index and row["gene_b"] in string_similarity.index:
        string_similarity.loc[row["gene_a"], row["gene_b"]] = row["score"]
        string_similarity.loc[row["gene_b"], row["gene_a"]] = row["score"]

# how many of the 50 genes have zero recorded association with every other selected gene
n_isolated = (string_similarity.sum(axis=1) == 0).sum()
string_similarity.shape, n_isolated

((50, 50), np.int64(4))

## Kernel-weighted similarity prediction

For a query gene in a given condition: look up its STRING similarity to each pool gene, raise `similarity + a small floor` to a tunable power ("temperature"), normalize to weights, and predict the query's RNA fingerprint as the weighted average of the pool genes' *true* RNA fingerprints. The floor (`EPSILON`) guarantees every gene gets at least a small, near-uniform weight even if its raw STRING similarity is exactly 0 to everyone in the pool -- without it, a query gene with zero measured association to any pool gene would get an all-zero weight vector (0 divided by 0).

Temperature plays the same role bandwidth played in model 4: temperature = 0 makes every weight exactly equal (`anything ** 0 == 1`), which is mathematically identical to the Task3_02 baseline; higher temperature increasingly concentrates weight on the most functionally similar training genes.

In [5]:
EPSILON = 0.01

def similarity_predict(query_gene, condition, temperature, pool_genes):
    """Predict an RNA fingerprint as a STRING-similarity-weighted average of pool genes' true RNA fingerprints."""
    # exclude the query gene itself from the pool used to compute the average
    fit_genes = [g for g in pool_genes if g != query_gene]

    # STRING similarity from the query gene to each pool gene (0 if no recorded evidence)
    sims = np.array([string_similarity.loc[query_gene, g] for g in fit_genes])

    weights = (sims + EPSILON) ** temperature
    weights /= weights.sum()

    # weighted average of the pool genes' own true RNA fingerprints
    fit_fingerprints = np.vstack([pert_FC_selected.loc[(g, condition)].values for g in fit_genes])
    pred_fingerprint = weights @ fit_fingerprints
    return pred_fingerprint

## Choosing the temperature via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate temperatures. This never touches the 10 held-out test genes -- the temperature is fixed before we ever look at them.

In [6]:
candidate_temperatures = [0, 1, 2, 4, 8, 16, 32, 64]

cv_mse_by_temperature = {}
for temperature in candidate_temperatures:
    squared_errors = []
    for cond in conditions:
        for gene in train_40:
            # similarity_predict excludes the query gene itself from the pool, so this is a genuine leave-one-out prediction
            pred = similarity_predict(gene, cond, temperature, train_40)
            true = pert_FC_selected.loc[(gene, cond)].values
            squared_errors.append(np.mean((true - pred) ** 2))
    cv_mse_by_temperature[temperature] = np.mean(squared_errors)

best_temperature = min(cv_mse_by_temperature, key=cv_mse_by_temperature.get)
cv_mse_by_temperature, best_temperature

({0: np.float64(0.0022219808588258513),
  1: np.float64(0.0024247071401435555),
  2: np.float64(0.0028347620087975376),
  4: np.float64(0.003310759201542273),
  8: np.float64(0.003784455927332312),
  16: np.float64(0.00406743844035828),
  32: np.float64(0.0042423428119214455),
  64: np.float64(0.004306103610655609)},
 0)

## Predict the held-out test genes and evaluate

Use the chosen temperature to predict each of the 10 held-out genes' RNA fingerprint from their STRING similarity to the 40 training genes, then evaluate with the same metrics used for the other models so results are directly comparable.

In [7]:
# predict each held-out (gene, condition) pair from the 40 training genes, using the CV-chosen temperature
similarity_predictions = {
    (gene, cond): similarity_predict(gene, cond, best_temperature, train_40)
    for cond in conditions
    for gene in test_10
}


def evaluate_predictions(true_df, predictions_by_row):
    """Compare each true fingerprint against its predicted fingerprint (looked up per row)."""
    records = []
    for (pert, cond), true_fc in true_df.iterrows():
        pred_fc = predictions_by_row[(pert, cond)]
        pearson_r, _ = pearsonr(true_fc, pred_fc)
        spearman_r, _ = spearmanr(true_fc, pred_fc)
        mse = np.mean((true_fc - pred_fc) ** 2)
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        })
    return pd.DataFrame(records)


similarity_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], similarity_predictions)
similarity_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.828347,0.457212,0.001508
1,KCNN4,IFNγ,0.844713,0.399161,0.001100
2,KCNN4,Co-culture,0.862499,0.339077,0.001128
3,TIMM50,Control,0.741626,0.392917,0.002767
4,TIMM50,IFNγ,0.624917,0.294308,0.003312
5,TIMM50,Co-culture,0.698907,0.194893,0.003830
6,TXNDC17,Control,0.805104,0.492349,0.004353
7,TXNDC17,IFNγ,0.620988,0.446131,0.004324
8,TXNDC17,Co-culture,0.762693,0.396490,0.004462
9,CORO1A,Control,0.806694,0.375133,0.001195


In [8]:
metrics = ["pearson_r", "spearman_r", "mse"]

# per-condition breakdown (n=10 genes each) -- for biological interpretation
per_condition = similarity_eval.groupby("condition")[metrics].agg(["mean", "std"])

# pooled across all held-out (gene, condition) pairs (n=30) -- single headline number, comparable to the other models
overall = similarity_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.592829  0.498193   0.267087  0.107349  0.003967  0.005891
Control     0.754654  0.137720   0.398926  0.117838  0.002500  0.001594
IFNγ        0.717705  0.254562   0.379795  0.096609  0.002430  0.001927

In [9]:
overall

,pearson_r,spearman_r,mse
mean,0.688396,0.348603,0.002966
std,0.328608,0.119509,0.003637


## Discussion (initial draft -- please rewrite)

**What this notebook does:** Predicts a held-out gene's RNA fingerprint as a similarity-weighted average of the 40 training genes' own true fingerprints, exactly like model 4, but the similarity source here is STRING's functional/physical-association score between genes -- static prior biological knowledge (pathway co-membership, physical interaction, co-expression across public datasets, text-mining), not anything measured in our own experiment. We first checked KEGG pathway co-membership, but its coverage of our specific 50 genes was too sparse (35/50 genes annotated, only 6/10 test genes), so we switched to STRING, which covers 46/50 genes (all 10 test genes). STRING scores were fetched once via its public API and cached locally (`task3_string_similarity.csv`) so the notebook doesn't depend on live internet on every rerun.

**Results:** LOOCV finds *no* benefit from sharpening the weights toward more STRING-similar genes at all -- validation MSE increases monotonically from temperature=0 (0.002222) to temperature=64 (~0.0042+), meaning the cross-validated-optimal choice is temperature=0, i.e. a perfectly uniform average over all 40 training genes. Since that's mathematically identical to the Task3_02 baseline's prediction, the final test metrics come out *bit-for-bit identical* to the baseline: Pearson r = 0.688396, Spearman r = 0.348603, MSE = 0.002966.

**Interpretation:** unlike the two protein-log2FC-based models (3 and 4), which both showed a genuine (if shallow) interior CV minimum, general functional/physical-association knowledge about a gene shows no detectable relationship to how similar its *specific RNA knockout fingerprint* is to another gene's. This is a meaningful negative result: prior knowledge about which genes interact or share a pathway in general does not predict which genes produce similar transcriptional knockout signatures in this specific melanoma perturbation screen. It reinforces a pattern across all five models so far: signal that plausibly helps (this experiment's own protein readout) helps a little; signal that's about the gene in the abstract (control-cell expression statistics, external functional-association databases) helps not at all.